# 07-2 — Streamlit evaluation dashboard

Launch an interactive web dashboard for the charts and reports created by Notebooks 04 and 07. This notebook does not train or evaluate a model; it only reads saved report files.

## 1. Mount the project

Connect Google Drive and select the JobAI project folder containing the dashboard application and evaluation reports.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.chdir("/content/drive/MyDrive/JobAI")
os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"
print("project:", os.getcwd())

## 2. Install the lightweight dashboard packages

Install Streamlit and Plotly plus the same pinned model stack used by Notebook 06. Chart mode works on CPU; chat mode requires a GPU because it loads the saved quantized base model and LoRA adapter.

In [ ]:
%pip install --quiet streamlit plotly -r requirements-train-colab.txt
%pip install --quiet --upgrade "bitsandbytes>=0.46.1"

## 3. Check available reports

Confirm that the baseline report exists. If Notebook 07 has run, the dashboard will automatically add model comparisons, pilot decisions, and prediction diagnostics.

In [ ]:
from pathlib import Path
REPO = Path(os.environ["JOBAI_REPO"])
REPORTS = REPO / "reports"
APP_PATH = REPO / "apps" / "streamlit_evaluation_dashboard.py"
assert APP_PATH.is_file(), f"Dashboard application missing: {APP_PATH}"
assert (REPORTS / "baselines.csv").is_file(), "Run Notebook 04 before opening the dashboard"
for name in ["baselines.csv", "model_vs_baselines.csv", "model_pilot_decision.csv", "finetuned_test_predictions.csv"]:
    path = REPORTS / name
    print(("READY  " if path.is_file() else "MISSING"), name)

## 4. Launch Streamlit

Start Streamlit in the background and open port 8501 through Colab's authenticated proxy. Rerunning this cell stops the earlier process created by this notebook before starting a fresh dashboard.

In [ ]:
import importlib.metadata, subprocess, sys, time
from urllib.request import urlopen

bnb_version = importlib.metadata.version("bitsandbytes")
import bitsandbytes as bnb
print("Python executable:", sys.executable)
print("bitsandbytes    :", bnb_version)
if "STREAMLIT_PROCESS" in globals() and STREAMLIT_PROCESS.poll() is None:
    STREAMLIT_PROCESS.terminate()
    STREAMLIT_PROCESS.wait(timeout=10)
LOG_PATH = REPORTS / "streamlit_dashboard.log"
LOG_HANDLE = open(LOG_PATH, "w", encoding="utf-8")
STREAMLIT_PROCESS = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", str(APP_PATH), "--server.port=8501", "--server.address=0.0.0.0", "--server.headless=true", "--server.enableCORS=false", "--server.enableXsrfProtection=false", "--browser.gatherUsageStats=false"],
    cwd=REPO, stdout=LOG_HANDLE, stderr=subprocess.STDOUT, env=os.environ.copy(),
)
for _ in range(30):
    try:
        if urlopen("http://127.0.0.1:8501/_stcore/health", timeout=1).read() == b"ok":
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(f"Streamlit did not start. Inspect {LOG_PATH}")
print("Streamlit is ready. Log:", LOG_PATH)
from google.colab import output
from google.colab.output import eval_js
from IPython.display import HTML, display
proxy_url = eval_js("google.colab.kernel.proxyPort(8501)")
display(HTML(f'<p><a href="{proxy_url}" target="_blank">Open the JobAI dashboard in a new tab</a></p>'))
output.serve_kernel_port_as_iframe(8501, height=900)

## How to use the dashboard

Choose **Evaluation charts** or **Chat with fine-tuned model** from the sidebar. Chart mode compares saved metrics. Chat mode loads the adapter from Notebook 06 and works best when your question supplies eight quarterly vacancy values, a forecast horizon, latest value, and scale. After rerunning Notebook 07, refresh the page to load new reports. Stop the Colab runtime when finished to avoid consuming compute units.